# 9.17 — In-Context Learning

In-context learning (ICL) is the behavior where a Transformer uses examples placed in the prompt to infer the current task during the forward pass. No weights change; the context acts like a tiny temporary dataset, and attention turns similarity between the query and demonstrations into a weighted prediction.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build in-context learning one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so the prompt is not a magic spell. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays + linear algebra for attention and toy regression.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for synthetic examples.

### 1. A prompt is a temporary training set

In-context learning starts by putting demonstrations in the context window: pairs \((x_i,y_i)\) followed by a new query \(x\). The model is not trained again; it must infer the rule from the visible examples. In this toy task, the hidden rule is almost linear: larger `x` usually means larger `y`.

In [ ]:
x_demo_w = np.array([-2.0, -1.0, 0.5, 1.5])  # demonstration inputs inside the prompt.
y_demo_w = 2.0 * x_demo_w + 1.0              # demonstration labels supplied as text tokens in real ICL.
x_query_w = 1.0                              # the new input whose output we want.

print("demonstrations:", list(zip(x_demo_w, y_demo_w)))  # inspect the temporary dataset.
print("query x:", x_query_w)                             # inspect the held-out query.

▶ What you'll see: four example pairs and one query; the examples are the only task evidence available at inference time.

In [ ]:
plt.figure(figsize=(4.4, 3.2))
plt.scatter(x_demo_w, y_demo_w, s=80, color="steelblue", label="context examples")
plt.axvline(x_query_w, color="crimson", linestyle="--", label="query x")
plt.title("1: demonstrations define the task"); plt.xlabel("x"); plt.ylabel("y")
plt.legend(); plt.show()

▶ What you'll see: the query sits among labeled examples, so a similarity-based forward pass has evidence to interpolate.

*Why it's done this way:* ICL is easiest to reason about as nonparametric prediction. The examples are not stored in learned weights; they are stored in activations for this one forward pass, so removing the prompt removes the task evidence.

### 2. Attention converts similarity into weights

Attention scores each demonstration by similarity to the query, then softmax-normalizes those scores into weights \(\alpha_i\). The core arithmetic is

$$\alpha_i=\frac{e^{s(x,x_i)}}{\sum_j e^{s(x,x_j)}}.$$

Large scores become large weights, but every weight is positive and the weights sum to 1, so the output is an average rather than an unbounded sum.

In [ ]:
scores_w = np.array([2.0, 1.0, 0.0])  # worked lesson similarities over three demonstrations.
shifted_w = scores_w - np.max(scores_w)  # subtract max for stable softmax without changing probabilities.
weights_w = np.exp(shifted_w) / np.exp(shifted_w).sum()  # convert logits to attention mass.

print("scores:", scores_w)
print("softmax weights:", np.round(weights_w, 3))

assert np.allclose(np.round(weights_w, 3), [0.665, 0.245, 0.090])

▶ What you'll see: scores `[2, 1, 0]` become attention weights `[0.665, 0.245, 0.090]`.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["demo 0", "demo 1", "demo 2"], weights_w, color="teal")
plt.ylabel("attention weight"); plt.title("2: softmax turns scores into weights")
plt.ylim(0, 1); plt.show()

▶ What you'll see: the most similar demonstration gets most, but not all, of the attention mass.

*Why it's done this way:* softmax is a differentiable nearest-neighbor selector. It preserves rank order from the scores while normalizing scale, so downstream value averaging stays on the same scale as the labels.

### 3. Attention as implicit regression

If the values are labels, attention becomes kernel regression: predict by averaging labels with similarity weights,

$$\hat y=\sum_i \alpha_i y_i.$$

This is the simplest mathematical picture of ICL: query-key matching estimates which examples matter, and value averaging transfers their outputs.

In [ ]:
labels_w = np.array([1.0, 0.0, 1.0])  # class labels attached to the three demonstrations.
pred_prob_w = float(weights_w @ labels_w)  # attention-weighted label average.

print("weighted label probability:", round(pred_prob_w, 3))
print("predicted class:", int(pred_prob_w >= 0.5))

assert round(pred_prob_w, 3) == 0.755

▶ What you'll see: `0.665×1 + 0.245×0 + 0.090×1 = 0.755`, so the predicted class is 1.

In [ ]:
dists_w = -np.abs(x_query_w - x_demo_w)  # closer x values get less-negative scores.
alphas_reg_w = np.exp(dists_w - np.max(dists_w)); alphas_reg_w /= alphas_reg_w.sum()
y_hat_reg_w = float(alphas_reg_w @ y_demo_w)  # Nadaraya-Watson style attention regression.

print("query-to-demo scores:", np.round(dists_w, 3))
print("attention-regression y_hat:", round(y_hat_reg_w, 3))

▶ What you'll see: nearby examples receive most of the mass and their labels interpolate a prediction for the query.

In [ ]:
plt.figure(figsize=(4.6, 3.2))
plt.scatter(x_demo_w, y_demo_w, s=70, color="steelblue", label="examples")
plt.scatter([x_query_w], [y_hat_reg_w], s=90, color="crimson", label="ICL prediction")
for xi_w, yi_w, ai_w in zip(x_demo_w, y_demo_w, alphas_reg_w):
    plt.plot([xi_w, x_query_w], [yi_w, y_hat_reg_w], color="gray", alpha=ai_w + 0.1)
plt.title("3: attention performs implicit regression"); plt.xlabel("x"); plt.ylabel("y")
plt.legend(); plt.show()

▶ What you'll see: thick connections come from high-weight nearby examples; the prediction is their weighted average.

*Why it's done this way:* the formula is regression without gradient descent at test time. Attention supplies a data-dependent kernel \(\alpha_i\), and the value stream supplies the targets, so the forward pass can implement a temporary learner.

### 4. A toy causal self-attention block can copy labels forward

A Transformer does not receive `(x, y)` tables directly; it receives tokens. We can still build a tiny hand-designed self-attention layer where the query token attends back to demonstration tokens. Keys encode input `x`, values encode label `y`, and the causal mask prevents looking into the future.

In [ ]:
tokens_x_w = np.array([-2.0, -1.0, 0.5, 1.5, x_query_w])  # four demos followed by query.
tokens_y_w = np.array([-3.0, -1.0, 2.0, 4.0, 0.0])        # query label is blank, so its value is 0.
K_w = tokens_x_w[:, None]                                  # keys: input coordinate.
Q_w = tokens_x_w[:, None]                                  # queries: input coordinate.
V_w = tokens_y_w[:, None]                                  # values: labels to average.

print("sequence length:", len(tokens_x_w), "query position:", len(tokens_x_w) - 1)

▶ What you'll see: a five-token sequence where the last token is the query and earlier tokens carry labels.

In [ ]:
scale_w = 1.0
raw_scores_w = (Q_w @ K_w.T) * scale_w  # dot-product attention scores.
mask_w = np.triu(np.ones_like(raw_scores_w, dtype=bool), k=1)  # future positions are illegal.
masked_scores_w = np.where(mask_w, -1e9, raw_scores_w)  # causal mask.
row_w = masked_scores_w[-1]  # scores from the query token to all previous tokens.
a_w = np.exp(row_w - np.max(row_w)); a_w /= a_w.sum()  # attention weights for query row.

print("query attention:", np.round(a_w, 3))
print("sum:", round(float(a_w.sum()), 3))

assert round(float(a_w.sum()), 3) == 1.0

▶ What you'll see: the query distributes attention over earlier examples and itself, with weights summing to 1.

In [ ]:
out_w = float(a_w @ V_w[:, 0])  # self-attention output at the query token.

print("attention output at query:", round(out_w, 3))

plt.figure(figsize=(4.6, 3))
plt.bar(["d0", "d1", "d2", "d3", "query"], a_w, color="darkorange")
plt.title("4: causal attention weights at the query"); plt.ylabel("weight"); plt.show()

▶ What you'll see: only visible tokens contribute; the query's blank value can dilute the label average if it receives mass.

*Why it's done this way:* causal masking is what makes language modeling legal: each position predicts from the past. For ICL, the query token can look backward to examples, and attention's value average is the mechanism that transports labels into the query representation.

### 5. Adding a more relevant example changes the prediction immediately

Because the examples live in the prompt, editing the prompt edits the computation. A new demonstration with score 3 has softmax weight 0.644 in the lesson arithmetic, which can dominate earlier evidence without changing any parameter.

In [ ]:
scores_plus_w = np.array([2.0, 1.0, 0.0, 3.0])  # add a more similar last demonstration.
weights_plus_w = np.exp(scores_plus_w - np.max(scores_plus_w)); weights_plus_w /= weights_plus_w.sum()

print("new weights:", np.round(weights_plus_w, 3))

assert round(float(weights_plus_w[-1]), 3) == 0.644

▶ What you'll see: the added example receives about 64.4% of the attention mass.

In [ ]:
labels_plus_w = np.array([1.0, 0.0, 1.0, 0.0])  # the new similar example has label 0.
pred_plus_w = float(weights_plus_w @ labels_plus_w)

print("old prediction:", round(pred_prob_w, 3), "new prediction:", round(pred_plus_w, 3))

plt.figure(figsize=(4.5, 3))
plt.bar(["before", "after adding demo"], [pred_prob_w, pred_plus_w], color=["steelblue", "crimson"])
plt.ylim(0, 1); plt.ylabel("predicted P(class=1)"); plt.title("5: prompt edits change behavior")
plt.show()

▶ What you'll see: a single more similar counterexample can flip or strongly reduce the predicted class probability.

*Why it's done this way:* ICL is conditional computation. The prompt examples are arguments to the forward pass, so adding a high-similarity example changes the normalized weights and therefore changes the answer immediately.

### 6. Context length and recency are real terms, not vibes

The context window is finite. If each demonstration costs tokens, more examples leave fewer query or reasoning tokens. Also, positional effects can add a recency bias to attention logits, changing the weights even when semantic similarity is unchanged.

In [ ]:
window_w = 8
example_tokens_w = 2
n_examples_w = 3
remaining_w = window_w - n_examples_w * example_tokens_w

print("remaining query tokens:", remaining_w)

assert remaining_w == 2

▶ What you'll see: three 2-token examples consume 6 of 8 tokens, leaving only 2 query tokens.

In [ ]:
base_recency_w = np.array([2.0, 1.0, 0.0])
biased_recency_w = base_recency_w.copy(); biased_recency_w[-1] += 0.5
w_base_recency = np.exp(base_recency_w - np.max(base_recency_w)); w_base_recency /= w_base_recency.sum()
w_biased_recency = np.exp(biased_recency_w - np.max(biased_recency_w)); w_biased_recency /= w_biased_recency.sum()

print("last logit before/after:", base_recency_w[-1], biased_recency_w[-1])
print("last weight before/after:", round(w_base_recency[-1], 3), round(w_biased_recency[-1], 3))

assert biased_recency_w[-1] == 0.5

▶ What you'll see: adding `+0.5` to the last example's logit raises its attention weight.

In [ ]:
xpos_w = np.arange(3)
plt.figure(figsize=(4.6, 3))
plt.bar(xpos_w - 0.18, w_base_recency, width=0.36, label="semantic only", color="gray")
plt.bar(xpos_w + 0.18, w_biased_recency, width=0.36, label="+ recency", color="seagreen")
plt.xticks(xpos_w, ["first", "middle", "last"]); plt.ylabel("attention weight")
plt.title("6: recency changes attention mass"); plt.legend(); plt.show()

▶ What you'll see: the last example gains weight even though only a positional logit changed.

*Why it's done this way:* order sensitivity is not mystical; examples are tokens with positions. If position enters the score, then permutation changes logits, logits change softmax weights, and weights change predictions.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses a
> handful of small numbers, prints every intermediate value with an inline `# ->` showing the
> result, draws one picture, and ends with an `assert` that pins the answer.

### ✍️ Toy 1 · Prompt examples act like a temporary dataset

The prompt carries examples for this one forward pass. A tiny line rule can be inferred from the
visible examples without changing any model weights.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


t1_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t1_x = np.array([0.0, 1.0, 2.0])                      # -> [0.0, 1.0, 2.0]

print("context x values:", t1_x.tolist())             # -> [0.0, 1.0, 2.0]

t1_y = np.array([1.0, 3.0, 5.0])                      # -> [1.0, 3.0, 5.0]

print("context y values:", t1_y.tolist())             # -> [1.0, 3.0, 5.0]

t1_query = 3.0                                        # -> 3.0

print("query x:", t1_query)                           # -> 3.0

t1_dx = t1_x[-1] - t1_x[0]                            # -> 2.0

print("x span:", round(float(t1_dx), 3))              # -> 2.0

t1_dy = t1_y[-1] - t1_y[0]                            # -> 4.0

print("y span:", round(float(t1_dy), 3))              # -> 4.0

t1_slope = t1_dy / t1_dx                              # -> 2.0

print("in-context slope:", round(float(t1_slope), 3)) # -> 2.0

t1_intercept = t1_y[0] - t1_slope * t1_x[0]           # -> 1.0

print("in-context intercept:", round(float(t1_intercept), 3))  # -> 1.0

t1_pred = t1_slope * t1_query + t1_intercept          # -> 7.0

print("query prediction:", round(float(t1_pred), 3))  # -> 7.0

assert round(float(t1_pred), 3) == 7.0

plt.figure(figsize=(4.8, 2.8))
plt.scatter(t1_x, t1_y, s=70, color="#4c78a8", label="prompt examples")
plt.scatter([t1_query], [t1_pred], s=90, color="#e15759", label="query prediction")
plt.plot([t1_x[0], t1_query], [t1_y[0], t1_pred], color="gray", linestyle="--")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Toy 1 · temporary dataset")
plt.legend()
plt.show()

▶ What you'll see: the query prediction extends the rule visible in the prompt examples.

### ✍️ Toy 2 · Attention softmax normalizes similarities

Similarity scores become attention weights by subtracting the max, exponentiating, and normalizing.
The weights are positive and sum to one.

In [ ]:
import numpy as np


t2_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t2_scores = np.array([0.0, 1.0, 1.5])                 # -> [0.0, 1.0, 1.5]

print("similarity scores:", t2_scores.tolist())       # -> [0.0, 1.0, 1.5]

t2_shifted = t2_scores - np.max(t2_scores)            # -> [-1.5, -0.5, 0.0]

print("shifted scores:", t2_shifted.tolist())         # -> [-1.5, -0.5, 0.0]

t2_exp = np.exp(t2_shifted)                           # -> [0.223, 0.607, 1.0]

print("exp shifted:", np.round(t2_exp, 3).tolist())   # -> [0.223, 0.607, 1.0]

t2_weights = t2_exp / t2_exp.sum()                    # -> [0.122, 0.331, 0.547]

print("attention weights:", np.round(t2_weights, 3).tolist())  # -> [0.122, 0.331, 0.547]

t2_sum = float(t2_weights.sum())                      # -> 1.0

print("weight sum:", round(t2_sum, 3))                # -> 1.0

assert np.allclose(np.round(t2_weights, 3), [0.122, 0.331, 0.547])

plt.figure(figsize=(4.8, 2.8))
plt.bar(["demo 0", "demo 1", "demo 2"], t2_weights, color="#4c78a8")
plt.ylim(0, 1)
plt.ylabel("attention weight")
plt.title("Toy 2 · softmax weights")
plt.show()

▶ What you'll see: the largest similarity receives the most attention, but the other examples still keep mass.

### ✍️ Toy 3 · Attention-weighted labels make a prediction

Once attention weights exist, prediction is just a weighted average of the labels attached to the
context examples.

In [ ]:
import numpy as np


t3_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t3_weights = np.array([0.122, 0.331, 0.547])          # -> [0.122, 0.331, 0.547]

print("attention weights:", t3_weights.tolist())      # -> [0.122, 0.331, 0.547]

t3_labels = np.array([0.0, 1.0, 1.0])                 # -> [0.0, 1.0, 1.0]

print("demo labels:", t3_labels.tolist())             # -> [0.0, 1.0, 1.0]

t3_contrib = t3_weights * t3_labels                   # -> [0.0, 0.331, 0.547]

print("weighted contributions:", np.round(t3_contrib, 3).tolist())  # -> [0.0, 0.331, 0.547]

t3_pred = float(t3_contrib.sum())                     # -> 0.878

print("predicted probability:", round(t3_pred, 3))    # -> 0.878

t3_class = int(t3_pred >= 0.5)                        # -> 1

print("predicted class:", t3_class)                   # -> 1

assert round(t3_pred, 3) == 0.878 and t3_class == 1

plt.figure(figsize=(4.8, 2.8))
plt.bar(["demo 0", "demo 1", "demo 2"], t3_contrib, color=["gray", "#59a14f", "#59a14f"])
plt.ylabel("weight × label")
plt.title("Toy 3 · weighted label average")
plt.show()

▶ What you'll see: the two positive-label examples contribute `0.878` total probability, so class `1` wins.

### ✍️ Toy 4 · Causal masking blocks future tokens

At a query position, causal self-attention can use previous tokens and itself, but future tokens get
zero attention mass.

In [ ]:
import numpy as np


t4_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t4_raw_scores = np.array([0.0, 1.0, 2.0, 3.0])        # -> [0.0, 1.0, 2.0, 3.0]

print("raw query scores:", t4_raw_scores.tolist())    # -> [0.0, 1.0, 2.0, 3.0]

t4_allowed = np.array([True, True, True, False])      # -> [True, True, True, False]

print("causal allowed mask:", t4_allowed.tolist())    # -> [True, True, True, False]

t4_masked = np.where(t4_allowed, t4_raw_scores, -9.0) # -> [0.0, 1.0, 2.0, -9.0]

print("masked scores:", t4_masked.tolist())           # -> [0.0, 1.0, 2.0, -9.0]

t4_shifted = t4_masked - np.max(t4_masked)            # -> [-2.0, -1.0, 0.0, -11.0]

print("shifted masked scores:", t4_shifted.tolist())  # -> [-2.0, -1.0, 0.0, -11.0]

t4_exp = np.where(t4_allowed, np.exp(t4_shifted), 0.0)  # -> [0.135, 0.368, 1.0, 0.0]

print("masked exponentials:", np.round(t4_exp, 3).tolist())  # -> [0.135, 0.368, 1.0, 0.0]

t4_weights = t4_exp / t4_exp.sum()                    # -> [0.09, 0.245, 0.665, 0.0]

print("query attention weights:", np.round(t4_weights, 3).tolist())  # -> [0.09, 0.245, 0.665, 0.0]

t4_values = np.array([0.0, 1.0, 2.0, 9.0])            # -> [0.0, 1.0, 2.0, 9.0]

print("token values:", t4_values.tolist())            # -> [0.0, 1.0, 2.0, 9.0]

t4_output = float(t4_weights @ t4_values)             # -> 1.575

print("attention output:", round(t4_output, 3))       # -> 1.575

assert round(float(t4_weights[-1]), 3) == 0.0 and round(t4_output, 3) == 1.575

plt.figure(figsize=(4.8, 2.8))
plt.bar(["past 0", "past 1", "query", "future"], t4_weights, color=["#4c78a8", "#4c78a8", "#59a14f", "#e15759"])
plt.ylabel("attention weight")
plt.title("Toy 4 · causal mask")
plt.show()

▶ What you'll see: the future token has zero weight even though its raw score was largest.

### ✍️ Toy 5 · Adding a relevant demonstration changes the answer

Because prompt examples are inputs, adding one high-similarity counterexample immediately changes
the normalized weights and the prediction.

In [ ]:
import numpy as np


t5_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t5_old_scores = np.array([1.0, 0.0])                  # -> [1.0, 0.0]

print("old scores:", t5_old_scores.tolist())          # -> [1.0, 0.0]

t5_old_labels = np.array([1.0, 0.0])                  # -> [1.0, 0.0]

print("old labels:", t5_old_labels.tolist())          # -> [1.0, 0.0]

t5_old_exp = np.exp(t5_old_scores - np.max(t5_old_scores))  # -> [1.0, 0.368]

print("old exp:", np.round(t5_old_exp, 3).tolist())   # -> [1.0, 0.368]

t5_old_weights = t5_old_exp / t5_old_exp.sum()        # -> [0.731, 0.269]

print("old weights:", np.round(t5_old_weights, 3).tolist())  # -> [0.731, 0.269]

t5_old_pred = float(t5_old_weights @ t5_old_labels)  # -> 0.731

print("old prediction:", round(t5_old_pred, 3))       # -> 0.731

t5_new_scores = np.array([1.0, 0.0, 2.0])             # -> [1.0, 0.0, 2.0]

print("new scores:", t5_new_scores.tolist())          # -> [1.0, 0.0, 2.0]

t5_new_labels = np.array([1.0, 0.0, 0.0])             # -> [1.0, 0.0, 0.0]

print("new labels:", t5_new_labels.tolist())          # -> [1.0, 0.0, 0.0]

t5_new_exp = np.exp(t5_new_scores - np.max(t5_new_scores))  # -> [0.368, 0.135, 1.0]

print("new exp:", np.round(t5_new_exp, 3).tolist())   # -> [0.368, 0.135, 1.0]

t5_new_weights = t5_new_exp / t5_new_exp.sum()        # -> [0.245, 0.09, 0.665]

print("new weights:", np.round(t5_new_weights, 3).tolist())  # -> [0.245, 0.09, 0.665]

t5_new_pred = float(t5_new_weights @ t5_new_labels)  # -> 0.245

print("new prediction:", round(t5_new_pred, 3))       # -> 0.245

assert t5_new_pred < t5_old_pred

plt.figure(figsize=(4.8, 2.8))
plt.bar(["before", "after adding demo"], [t5_old_pred, t5_new_pred], color=["#4c78a8", "#e15759"])
plt.ylim(0, 1)
plt.ylabel("P(class 1)")
plt.title("Toy 5 · prompt edit changes prediction")
plt.show()

▶ What you'll see: the new similar label-0 example cuts the class-1 prediction from `0.731` to `0.245`.

### ✍️ Toy 6 · Token budget and recency bias change weights

A finite context window limits how many demonstration tokens fit. A recency logit can also shift
attention toward later examples even when semantic scores are tied.

In [ ]:
import numpy as np


t6_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t6_window = 10                                        # -> 10

print("context window:", t6_window)                   # -> 10

t6_examples = 3                                       # -> 3

print("number of examples:", t6_examples)             # -> 3

t6_tokens_each = 2                                    # -> 2

print("tokens per example:", t6_tokens_each)          # -> 2

t6_used = t6_examples * t6_tokens_each                # -> 6

print("tokens used:", t6_used)                        # -> 6

t6_remaining = t6_window - t6_used                    # -> 4

print("tokens remaining:", t6_remaining)              # -> 4

t6_base_scores = np.array([1.0, 1.0, 1.0])            # -> [1.0, 1.0, 1.0]

print("base scores:", t6_base_scores.tolist())        # -> [1.0, 1.0, 1.0]

t6_recency_bonus = np.array([0.0, 0.2, 0.4])          # -> [0.0, 0.2, 0.4]

print("recency bonus:", t6_recency_bonus.tolist())    # -> [0.0, 0.2, 0.4]

t6_biased_scores = t6_base_scores + t6_recency_bonus  # -> [1.0, 1.2, 1.4]

print("biased scores:", np.round(t6_biased_scores, 3).tolist())  # -> [1.0, 1.2, 1.4]

t6_base_exp = np.exp(t6_base_scores - np.max(t6_base_scores))  # -> [1.0, 1.0, 1.0]

print("base exp:", np.round(t6_base_exp, 3).tolist()) # -> [1.0, 1.0, 1.0]

t6_base_weights = t6_base_exp / t6_base_exp.sum()     # -> [0.333, 0.333, 0.333]

print("base weights:", np.round(t6_base_weights, 3).tolist())  # -> [0.333, 0.333, 0.333]

t6_biased_exp = np.exp(t6_biased_scores - np.max(t6_biased_scores))  # -> [0.67, 0.819, 1.0]

print("biased exp:", np.round(t6_biased_exp, 3).tolist())  # -> [0.67, 0.819, 1.0]

t6_biased_weights = t6_biased_exp / t6_biased_exp.sum()  # -> [0.269, 0.329, 0.402]

print("biased weights:", np.round(t6_biased_weights, 3).tolist())  # -> [0.269, 0.329, 0.402]

assert t6_remaining == 4 and t6_biased_weights[-1] > t6_base_weights[-1]

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(3) - 0.18, t6_base_weights, width=0.36, label="base", color="gray")
plt.bar(np.arange(3) + 0.18, t6_biased_weights, width=0.36, label="+ recency", color="#59a14f")
plt.xticks(np.arange(3), ["first", "middle", "last"])
plt.ylabel("attention weight")
plt.title("Toy 6 · recency shifts mass")
plt.legend()
plt.show()

▶ What you'll see: three examples fit with four tokens left, and the last example gains attention from the recency bonus.


## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, vectorized attention, and tiny numerical checks.
import matplotlib.pyplot as plt # load Matplotlib so each concept can be inspected visually.
np.random.seed(0) # make all random examples reproducible across notebook runs.

def softmax(z): # define a stable softmax helper for attention logits.
    z = np.asarray(z, dtype=float) # convert logits to a float array.
    e = np.exp(z - np.max(z)) # subtract the maximum for numerical stability.
    return e / e.sum() # normalize exponentials into probabilities that sum to one.

def attention_predict(scores, values): # define attention as a weighted average over values.
    w = softmax(scores) # convert similarity scores into attention weights.
    return float(w @ np.asarray(values, dtype=float)), w # return prediction and weights for inspection.

def rbf_scores(x, xs, width=1.0): # define distance-based similarity for toy ICL regression.
    return -((np.asarray(xs, dtype=float) - float(x)) ** 2) / (2 * width ** 2) # negative squared distance logits.

def show_weights(weights, title): # define a small bar helper for attention distributions.
    plt.figure(figsize=(4, 3)) # create a compact figure.
    plt.bar(np.arange(len(weights)), weights, color="teal") # draw one bar per context example.
    plt.title(title) # title the plot.
    plt.xlabel("context example") # label the example axis.
    plt.ylabel("attention weight") # label the probability axis.
    plt.ylim(0, 1) # keep attention on a fixed scale.
    plt.show() # display the chart.

## 🟢 Basics (warm-up)

### Basic 1 — Turn similarities into attention

**Goal.** Convert three demonstration scores into weights, because ICL needs a way to decide which examples influence the query. We build it in 2 steps.

In [ ]:
scores_b1 = np.array([2.0, 1.0, 0.0]) # store similarity logits from the lesson mechanics.

print("scores:", scores_b1) # inspect raw, unnormalized evidence.

In [ ]:
weights_b1 = softmax(scores_b1) # normalize logits into attention weights.

print("weights:", np.round(weights_b1, 3)) # inspect the canonical softmax weights.

assert np.allclose(np.round(weights_b1, 3), [0.665, 0.245, 0.090]) # verify the lesson numbers.
show_weights(weights_b1, "Basic 1: softmax attention weights") # visualize which example dominates.

▶ What you'll see: the first example receives most of the attention mass, but the others still contribute.

👀 Takeaway: attention weights are normalized similarities, not hard nearest-neighbor choices.

### Basic 2 — Average labels with attention

**Goal.** Use attention weights to average labels, because the simplest ICL classifier is weighted voting over demonstrations. We build it in 2 steps.

In [ ]:
labels_b2 = np.array([1.0, 0.0, 1.0]) # define three demonstration labels.
weights_b2 = softmax(np.array([2.0, 1.0, 0.0])) # reuse the lesson attention pattern.

print("labels:", labels_b2) # inspect the values that attention will average.

In [ ]:
pred_b2 = float(weights_b2 @ labels_b2) # compute attention-weighted class probability.

print("P(class=1):", round(pred_b2, 3)) # inspect the predicted probability.

assert round(pred_b2, 3) == 0.755 # verify the worked weighted average.
plt.figure(figsize=(4, 3)); plt.bar(["demo0", "demo1", "demo2"], weights_b2 * labels_b2, color="purple")
plt.title("Basic 2: weighted label contributions"); plt.ylabel("weight × label"); plt.show()

▶ What you'll see: positive-label examples contribute mass while the zero-label example contributes none.

👀 Takeaway: ICL can look like attention-weighted voting when values are labels.

### Basic 3 — Compute distance-based scores

**Goal.** Score examples by closeness to a query, because attention needs a similarity function before it can weight labels. We build it in 2 steps.

In [ ]:
xs_b3 = np.array([-2.0, -0.5, 0.5, 2.0]) # context inputs.
xq_b3 = 0.25 # query input.
scores_b3 = rbf_scores(xq_b3, xs_b3, width=1.0) # closer examples get larger logits.

print("scores:", np.round(scores_b3, 3)) # inspect distance-derived similarities.

In [ ]:
weights_b3 = softmax(scores_b3) # turn closeness scores into attention weights.

print("weights:", np.round(weights_b3, 3)) # inspect which context point is closest.

assert int(np.argmax(weights_b3)) == 2 # verify x=0.5 is the nearest and highest-weight example.
show_weights(weights_b3, "Basic 3: distance scores become attention") # visualize the distribution.

▶ What you'll see: the example at `x=0.5` receives the largest weight because it is nearest the query.

👀 Takeaway: similarity choice controls which demonstrations the forward pass trusts.

### Basic 4 — Do one in-context regression prediction

**Goal.** Predict a numeric output by averaging nearby example labels, because attention can implement a tiny regression model. We build it in 3 steps.

In [ ]:
xs_b4 = np.array([-1.0, 0.0, 1.0, 2.0]) # demonstration inputs.
ys_b4 = 3.0 * xs_b4 - 1.0 # demonstration labels from a hidden linear rule.
xq_b4 = 0.8 # query input near x=1.

print("examples:", list(zip(xs_b4, ys_b4))) # inspect the temporary dataset.

In [ ]:
scores_b4 = rbf_scores(xq_b4, xs_b4, width=0.8) # compute query-example similarities.
pred_b4, weights_b4 = attention_predict(scores_b4, ys_b4) # average labels with attention.

print("prediction:", round(pred_b4, 3), "true rule:", round(3 * xq_b4 - 1, 3)) # compare ICL to hidden rule.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.scatter(xs_b4, ys_b4, color="steelblue", s=70, label="examples")
plt.scatter([xq_b4], [pred_b4], color="crimson", s=90, label="ICL pred")
plt.title("Basic 4: attention regression"); plt.xlabel("x"); plt.ylabel("y")
plt.legend(); plt.show()

▶ What you'll see: the prediction lands near the line implied by the nearby context examples.

👀 Takeaway: attention regression interpolates from prompt examples without updating parameters.

### Basic 5 — Show no parameter update is needed

**Goal.** Separate prompt-conditioned prediction from training, because ICL changes activations but not model weights. We build it in 2 steps.

In [ ]:
params_b5 = np.array([0.4, -0.2]) # pretend these are fixed model parameters.
before_b5 = params_b5.copy() # save a copy before the prompt is processed.
context_scores_b5 = np.array([1.5, 0.2]) # prompt-specific evidence.

print("params before:", before_b5) # inspect fixed parameters.

In [ ]:
pred_b5, weights_b5 = attention_predict(context_scores_b5, np.array([10.0, -2.0])) # compute with context only.
after_b5 = params_b5.copy() # parameters are unchanged after the forward pass.

print("prediction:", round(pred_b5, 3), "params after:", after_b5) # inspect output and unchanged weights.

assert np.allclose(before_b5, after_b5) # verify no learning step occurred.

In [ ]:
plt.figure(figsize=(4.2, 3.0))
x_b5 = np.arange(len(params_b5))
width_b5 = 0.35
plt.bar(x_b5 - width_b5 / 2, before_b5, width_b5, label="before prompt")
plt.bar(x_b5 + width_b5 / 2, after_b5, width_b5, label="after prompt")
plt.xticks(x_b5, ["weight 0", "weight 1"])
plt.title("Basic 5: parameters stay unchanged"); plt.ylabel("parameter value")
plt.legend(); plt.show()

▶ What you'll see: paired bars overlap exactly, showing that the context changed the prediction without changing stored parameters.

▶ What you'll see: the output changes because context values are used, while the stored parameters stay identical.

👀 Takeaway: in-context learning is inference-time adaptation, not permanent training.

### Basic 6 — Add a more similar demonstration

**Goal.** Watch a new prompt example change attention mass, because ICL behavior is conditional on the current context. We build it in 2 steps.

In [ ]:
scores_b6 = np.array([2.0, 1.0, 0.0]) # original scores.
labels_b6 = np.array([1.0, 0.0, 1.0]) # original labels.
old_pred_b6, old_w_b6 = attention_predict(scores_b6, labels_b6) # original prompt prediction.

print("old prediction:", round(old_pred_b6, 3)) # inspect baseline behavior.

In [ ]:
new_scores_b6 = np.array([2.0, 1.0, 0.0, 3.0]) # append a highly similar example.
new_labels_b6 = np.array([1.0, 0.0, 1.0, 0.0]) # the appended example has the opposite label.
new_pred_b6, new_w_b6 = attention_predict(new_scores_b6, new_labels_b6) # recompute with the edited prompt.

print("new weights:", np.round(new_w_b6, 3), "new prediction:", round(new_pred_b6, 3)) # inspect changed behavior.

assert round(float(new_w_b6[-1]), 3) == 0.644 # verify the lesson's new-example weight.
plt.figure(figsize=(4, 3)); plt.bar(["old", "new"], [old_pred_b6, new_pred_b6], color=["gray", "red"])
plt.ylim(0, 1); plt.title("Basic 6: one new demo changes prediction"); plt.show()

▶ What you'll see: the new high-similarity example receives most attention and pulls the probability downward.

👀 Takeaway: prompt edits are computation edits in ICL.

### Basic 7 — Count context budget

**Goal.** Compute how many query tokens remain after demonstrations, because context length is a hard resource. We build it in 2 steps.

In [ ]:
window_b7 = 8 # total token budget.
tokens_per_example_b7 = 2 # each demonstration uses two tokens in this toy accounting.
examples_b7 = 3 # number of demonstrations inserted into the prompt.
used_b7 = tokens_per_example_b7 * examples_b7 # compute context used by examples.

print("used tokens:", used_b7) # inspect prompt cost.

In [ ]:
remaining_b7 = window_b7 - used_b7 # compute remaining budget for query and answer prefix.

print("remaining tokens:", remaining_b7) # inspect leftover capacity.

assert remaining_b7 == 2 # verify the lesson arithmetic 8 - 6 = 2.
plt.figure(figsize=(4, 3)); plt.bar(["used by demos", "remaining"], [used_b7, remaining_b7], color=["orange", "teal"])
plt.title("Basic 7: finite context budget"); plt.ylabel("tokens"); plt.show()

▶ What you'll see: examples can crowd out space needed for the query or reasoning.

👀 Takeaway: more demonstrations are useful only until they consume too much context.

### Basic 8 — Add recency bias to logits

**Goal.** Modify one attention logit by position, because prompt order can change predictions when positions enter the score. We build it in 2 steps.

In [ ]:
logits_b8 = np.array([2.0, 1.0, 0.0]) # semantic similarity logits.
biased_b8 = logits_b8.copy(); biased_b8[-1] += 0.5 # add a positional recency boost to the last example.

print("last logit before/after:", logits_b8[-1], biased_b8[-1]) # inspect the explicit score change.

In [ ]:
w0_b8 = softmax(logits_b8) # weights before recency.
w1_b8 = softmax(biased_b8) # weights after recency.

print("last weight before/after:", round(w0_b8[-1], 3), round(w1_b8[-1], 3)) # inspect changed attention mass.

assert biased_b8[-1] == 0.5 # verify the logit changed from 0 to 0.5.
plt.figure(figsize=(4, 3)); plt.plot(w0_b8, marker="o", label="before"); plt.plot(w1_b8, marker="o", label="after")
plt.title("Basic 8: recency shifts weights"); plt.legend(); plt.show()

▶ What you'll see: the last example gets more weight after the positional logit boost.

👀 Takeaway: order sensitivity follows directly from logits that include positional information.

### Basic 9 — Dilution from irrelevant examples

**Goal.** Add many weak examples and measure attention on the useful one, because long contexts can spread mass away from relevant evidence. We build it in 2 steps.

In [ ]:
strong_b9 = np.array([3.0]) # one highly relevant demonstration.
weak_b9 = np.zeros(9) # nine irrelevant demonstrations with neutral scores.
scores_b9 = np.concatenate([strong_b9, weak_b9]) # combine them into one context.
weights_b9 = softmax(scores_b9) # compute attention over all examples.

print("weight on strong demo:", round(float(weights_b9[0]), 3)) # inspect dilution.

In [ ]:
scores_less_b9 = np.array([3.0, 0.0]) # same strong example with only one distractor.
weights_less_b9 = softmax(scores_less_b9) # compute less-diluted attention.

print("strong weight with 1 distractor:", round(float(weights_less_b9[0]), 3)) # compare.

assert weights_less_b9[0] > weights_b9[0] # verify more distractors dilute attention.
plt.figure(figsize=(4, 3)); plt.bar(["1 distractor", "9 distractors"], [weights_less_b9[0], weights_b9[0]], color="crimson")
plt.title("Basic 9: irrelevant context dilutes attention"); plt.ylabel("weight on useful demo"); plt.show()

▶ What you'll see: the same strong example receives less mass when many irrelevant examples are present.

👀 Takeaway: long context helps only if retrieval or attention keeps relevant demonstrations salient.

### Basic 10 — Compare zero-shot and few-shot logits

**Goal.** Show how demonstrations can act like a logit boost, because prompting changes the conditional distribution. We build it in 2 steps.

In [ ]:
zero_logit_b10 = 1.0 # zero-shot logit for answer A over answer B.
few_shot_boost_b10 = 0.8 # demonstration-induced evidence for A.
p_zero_b10 = 1 / (1 + np.exp(-zero_logit_b10)) # convert logit to probability.

print("zero-shot P(A):", round(p_zero_b10, 3)) # inspect baseline probability.

In [ ]:
p_few_b10 = 1 / (1 + np.exp(-(zero_logit_b10 + few_shot_boost_b10))) # add prompt evidence and convert.

print("few-shot P(A):", round(p_few_b10, 3)) # inspect conditioned probability.

assert round(p_zero_b10, 3) == 0.731 and round(p_few_b10, 3) == 0.858 # verify related prompting mechanics.
plt.figure(figsize=(4, 3)); plt.bar(["zero-shot", "few-shot"], [p_zero_b10, p_few_b10], color=["gray", "teal"])
plt.ylim(0, 1); plt.title("Basic 10: demonstrations shift logits"); plt.show()

▶ What you'll see: adding demonstration evidence raises the model's probability for the supported answer.

👀 Takeaway: few-shot prompting can be viewed as conditioning that shifts output logits.

## 🟡 Easy

### Easy 1 — Implement a tiny ICL classifier

**Goal.** Classify a query from prompt examples, because ICL often uses demonstrations as temporary labeled data. We build it in 3 steps.

In [ ]:
x_e1 = np.array([-2.0, -1.0, 1.0, 2.0]) # demonstration inputs.
y_e1 = np.array([0.0, 0.0, 1.0, 1.0]) # demonstration labels: sign of x.
query_e1 = 0.6 # query near positive examples.

print("context labels:", y_e1) # inspect label evidence.

In [ ]:
scores_e1 = rbf_scores(query_e1, x_e1, width=1.0) # compute query-demo similarities.
prob_e1, weights_e1 = attention_predict(scores_e1, y_e1) # attention-weighted class probability.

print("weights:", np.round(weights_e1, 3), "P(class=1):", round(prob_e1, 3)) # inspect classifier internals.

In [ ]:
pred_class_e1 = int(prob_e1 >= 0.5) # threshold probability into a class.

print("predicted class:", pred_class_e1) # inspect final label.

assert pred_class_e1 == 1 # verify the positive query is classified positive.
show_weights(weights_e1, "Easy 1: classifier attention") # visualize evidence weights.

▶ What you'll see: positive demonstrations receive most weight, so the query is classified as 1.

👀 Takeaway: a few-shot classifier can be built from similarity, softmax, and weighted labels.

### Easy 2 — Fit a line in context with weighted least squares

**Goal.** Estimate a local linear rule from prompt examples, because attention can behave like implicit regression rather than simple averaging. We build it in 3 steps.

In [ ]:
x_e2 = np.array([-2.0, -1.0, 0.0, 1.0, 2.0]) # context inputs.
y_e2 = 1.5 * x_e2 + 0.5 # context labels from a linear rule.
query_e2 = 1.3 # query where we want a prediction.

print("true query value:", round(1.5 * query_e2 + 0.5, 3)) # inspect target from hidden rule.

In [ ]:
w_e2 = softmax(rbf_scores(query_e2, x_e2, width=1.2)) # locality weights around the query.
X_e2 = np.c_[np.ones_like(x_e2), x_e2] # design matrix for intercept and slope.
W_e2 = np.diag(w_e2) # diagonal weight matrix for weighted least squares.
beta_e2 = np.linalg.solve(X_e2.T @ W_e2 @ X_e2, X_e2.T @ W_e2 @ y_e2) # solve weighted normal equations.

print("estimated intercept/slope:", np.round(beta_e2, 3)) # inspect inferred task rule.

assert np.allclose(np.round(beta_e2, 3), [0.5, 1.5]) # verify exact recovery for noiseless linear data.

In [ ]:
pred_e2 = float(np.array([1.0, query_e2]) @ beta_e2) # predict query from inferred line.

print("weighted-LS ICL prediction:", round(pred_e2, 3)) # inspect prediction.

plt.figure(figsize=(4.6, 3)); plt.scatter(x_e2, y_e2, s=70); plt.scatter([query_e2], [pred_e2], color="red", s=90)
plt.title("Easy 2: in-context linear regression"); plt.xlabel("x"); plt.ylabel("y"); plt.show()

▶ What you'll see: the inferred line matches the demonstrations and predicts the query accurately.

👀 Takeaway: attention-like weights can support local least-squares regression inside the forward pass.

### Easy 3 — Build a one-head causal attention matrix

**Goal.** Apply a causal mask to self-attention, because a language model query may look backward to demonstrations but not forward. We build it in 3 steps.

In [ ]:
X_e3 = np.array([[1.0, 0.0], [0.5, 1.0], [1.0, 1.0]]) # three token embeddings.
Q_e3 = X_e3; K_e3 = X_e3; V_e3 = np.array([[1.0], [0.0], [1.0]]) # use embeddings as queries/keys and labels as values.

print("tokens:", X_e3.shape[0]) # inspect sequence length.

In [ ]:
logits_e3 = Q_e3 @ K_e3.T / np.sqrt(X_e3.shape[1]) # scaled dot-product attention logits.
mask_e3 = np.triu(np.ones_like(logits_e3, dtype=bool), k=1) # future positions.
logits_e3 = np.where(mask_e3, -1e9, logits_e3) # block future attention.
A_e3 = np.vstack([softmax(row_e3) for row_e3 in logits_e3]) # row-wise attention.

print("attention matrix:\n", np.round(A_e3, 3)) # inspect causal weights.

assert np.allclose(A_e3[0], [1.0, 0.0, 0.0]) # first token can only attend to itself.

In [ ]:
out_e3 = A_e3 @ V_e3 # average values at each position.

print("outputs:", np.round(out_e3[:, 0], 3)) # inspect label information copied forward.

plt.figure(figsize=(4, 3)); plt.imshow(A_e3, cmap="viridis"); plt.colorbar(label="weight")
plt.title("Easy 3: causal attention matrix"); plt.xlabel("key position"); plt.ylabel("query position"); plt.show()

▶ What you'll see: upper-triangular future weights are zero, and later tokens can mix earlier values.

👀 Takeaway: causal self-attention is the mechanism that lets query tokens read demonstrations to their left.

### Easy 4 — Measure prompt order sensitivity

**Goal.** Reverse example order under a recency bias, because examples are tokens with positions rather than an unordered set. We build it in 3 steps.

In [ ]:
labels_e4 = np.array([1.0, 0.0, 0.0]) # labels in original order.
semantic_e4 = np.array([1.0, 1.0, 1.0]) # equal semantic scores.
position_bias_e4 = np.array([0.0, 0.2, 0.5]) # later examples get a logit boost.

print("position bias:", position_bias_e4) # inspect order term.

In [ ]:
w_forward_e4 = softmax(semantic_e4 + position_bias_e4) # weights in original order.
pred_forward_e4 = float(w_forward_e4 @ labels_e4) # prediction for original order.
w_reverse_e4 = softmax(semantic_e4 + position_bias_e4) # same positional weights after reversing examples.
pred_reverse_e4 = float(w_reverse_e4 @ labels_e4[::-1]) # labels assigned to different positions.

print("forward/reverse predictions:", round(pred_forward_e4, 3), round(pred_reverse_e4, 3)) # inspect order effect.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["forward", "reversed"], [pred_forward_e4, pred_reverse_e4], color=["teal", "orange"])
plt.ylim(0, 1); plt.title("Easy 4: same set, different order"); plt.ylabel("P(class=1)"); plt.show()
assert pred_forward_e4 != pred_reverse_e4 # verify that order changed the result.

▶ What you'll see: reversing the same labels changes the prediction because different labels occupy high-bias positions.

👀 Takeaway: ICL can be order-sensitive whenever positional terms affect attention or logits.

### Easy 5 — Compare ICL to memorization

**Goal.** Test a query not present in the context, because useful ICL should infer a pattern rather than only copy exact strings. We build it in 3 steps.

In [ ]:
x_e5 = np.array([0.0, 1.0, 2.0, 3.0]) # context inputs from a simple rule.
y_e5 = x_e5 ** 2 # labels follow a nonlinear pattern.
query_e5 = 1.5 # query not exactly in the context.

print("is exact match present?", bool(np.any(x_e5 == query_e5))) # inspect memorization possibility.

In [ ]:
scores_e5 = rbf_scores(query_e5, x_e5, width=0.7) # similarity to surrounding examples.
pred_e5, weights_e5 = attention_predict(scores_e5, y_e5) # interpolate labels.

print("prediction:", round(pred_e5, 3), "nearest labels:", y_e5[1:3]) # inspect interpolation.

assert not np.any(x_e5 == query_e5) # verify no exact example was copied.

In [ ]:
plt.figure(figsize=(4.5, 3)); plt.scatter(x_e5, y_e5, color="steelblue", s=70); plt.scatter([query_e5], [pred_e5], color="red", s=90)
plt.title("Easy 5: generalizing between examples"); plt.xlabel("x"); plt.ylabel("y"); plt.show()

▶ What you'll see: the query prediction is built from nearby examples even though no exact match exists.

👀 Takeaway: ICL is most valuable when it infers a task pattern, not when it merely copies an exact demonstration.

## 🔴 Advanced

### Advanced 1 — Sweep temperature for attention sharpness

**Goal.** Vary softmax temperature, because it controls whether ICL behaves like nearest-neighbor lookup or broad averaging. We build it in 4 steps.

In [ ]:
scores_a1 = np.array([2.0, 1.0, 0.0]) # fixed semantic scores.
labels_a1 = np.array([1.0, 0.0, 1.0]) # fixed demonstration labels.
temps_a1 = np.array([0.25, 0.5, 1.0, 2.0]) # lower is sharper, higher is smoother.

print("temperatures:", temps_a1) # inspect sweep values.

In [ ]:
preds_a1 = [] # store predictions for each temperature.
entropies_a1 = [] # store attention entropy as a sharpness measure.
for temp_a1 in temps_a1: # loop over temperatures.
    w_a1 = softmax(scores_a1 / temp_a1) # temperature-scaled attention.
    preds_a1.append(float(w_a1 @ labels_a1)) # prediction under this attention distribution.
    entropies_a1.append(float(-np.sum(w_a1 * np.log(w_a1 + 1e-12)))) # entropy of the weights.

print("predictions:", np.round(preds_a1, 3)) # inspect output changes.
print("entropies:", np.round(entropies_a1, 3)) # inspect sharpness changes.

In [ ]:
assert entropies_a1[0] < entropies_a1[-1] # verify higher temperature spreads attention.
best_sharp_a1 = temps_a1[int(np.argmin(entropies_a1))] # identify sharpest setting.

print("sharpest temperature:", best_sharp_a1) # inspect nearest-neighbor-like setting.

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(temps_a1, preds_a1, marker="o", label="prediction")
plt.plot(temps_a1, entropies_a1, marker="s", label="attention entropy")
plt.title("Advanced 1: temperature controls ICL sharpness"); plt.xlabel("temperature"); plt.legend(); plt.show()

▶ What you'll see: low temperature concentrates on the top example; high temperature averages more broadly.

👀 Takeaway: attention temperature is a bias-variance knob for in-context prediction.

### Advanced 2 — Simulate retrieval before ICL

**Goal.** Select the most relevant demonstrations from a larger pool, because the context window should be filled with useful examples rather than arbitrary ones. We build it in 4 steps.

In [ ]:
pool_x_a2 = np.linspace(-4, 4, 17) # candidate demonstrations.
pool_y_a2 = 2 * pool_x_a2 + 1 # labels from a simple rule.
query_a2 = 1.2 # query point.
budget_a2 = 5 # context can hold only five examples.

print("pool size:", len(pool_x_a2), "budget:", budget_a2) # inspect retrieval pressure.

In [ ]:
similarity_a2 = -np.abs(pool_x_a2 - query_a2) # retrieval score by closeness.
chosen_a2 = np.argsort(similarity_a2)[::-1][:budget_a2] # top-k retrieved examples.
random_a2 = np.arange(budget_a2) # a bad baseline: first five examples.

print("retrieved x:", pool_x_a2[chosen_a2]) # inspect relevant context.
print("baseline x:", pool_x_a2[random_a2]) # inspect arbitrary context.

In [ ]:
pred_retrieved_a2, _ = attention_predict(rbf_scores(query_a2, pool_x_a2[chosen_a2], width=1.0), pool_y_a2[chosen_a2]) # ICL after retrieval.
pred_random_a2, _ = attention_predict(rbf_scores(query_a2, pool_x_a2[random_a2], width=1.0), pool_y_a2[random_a2]) # ICL with poor context.
truth_a2 = 2 * query_a2 + 1 # hidden rule value.

print("retrieved pred:", round(pred_retrieved_a2, 3), "random pred:", round(pred_random_a2, 3), "truth:", round(truth_a2, 3)) # compare.

assert abs(pred_retrieved_a2 - truth_a2) < abs(pred_random_a2 - truth_a2) # verify retrieval helps.

In [ ]:
plt.figure(figsize=(5, 3)); plt.scatter(pool_x_a2, pool_y_a2, c="lightgray", label="pool")
plt.scatter(pool_x_a2[chosen_a2], pool_y_a2[chosen_a2], c="teal", label="retrieved")
plt.axvline(query_a2, color="red", linestyle="--", label="query")
plt.title("Advanced 2: retrieval chooses context"); plt.legend(); plt.show()

▶ What you'll see: retrieved examples cluster around the query and yield a better in-context prediction.

👀 Takeaway: ICL quality depends strongly on which examples fit into the finite context window.

### Advanced 3 — Distinguish ICL from fine-tuning

**Goal.** Compare a prompt-only predictor with a gradient-updated parameter, because both adapt but only fine-tuning persists after the context is removed. We build it in 4 steps.

In [ ]:
x_train_a3 = np.array([0.0, 1.0, 2.0]) # tiny task examples.
y_train_a3 = 2.0 * x_train_a3 + 1.0 # labels from a line.
w_param_a3 = 0.0 # persistent model parameter for a deliberately tiny fine-tune demo.
query_a3 = 3.0 # query outside the prompt examples.

print("initial persistent weight:", w_param_a3) # inspect before fine-tuning.

In [ ]:
pred_icl_a3, _ = attention_predict(rbf_scores(query_a3, x_train_a3, width=1.0), y_train_a3) # prompt-only prediction.
grad_a3 = -2 * np.mean(x_train_a3 * (y_train_a3 - w_param_a3 * x_train_a3)) # gradient for y≈w*x.
w_tuned_a3 = w_param_a3 - 0.1 * grad_a3 # one persistent gradient step.

print("ICL prediction:", round(pred_icl_a3, 3), "tuned weight:", round(w_tuned_a3, 3)) # inspect both adaptations.

In [ ]:
pred_tuned_a3 = w_tuned_a3 * query_a3 # prediction from updated parameter.
context_removed_icl_a3 = 0.0 # no examples means this toy ICL rule has no evidence.

print("tuned prediction after context removed:", round(pred_tuned_a3, 3))
print("ICL after context removed:", context_removed_icl_a3)

assert w_tuned_a3 != w_param_a3 # verify fine-tuning changed a persistent value.

In [ ]:
plt.figure(figsize=(4.5, 3)); plt.bar(["ICL with prompt", "fine-tuned", "ICL no prompt"], [pred_icl_a3, pred_tuned_a3, context_removed_icl_a3], color=["teal", "orange", "gray"])
plt.title("Advanced 3: temporary vs persistent adaptation"); plt.ylabel("prediction"); plt.xticks(rotation=15); plt.show()

▶ What you'll see: ICL works only while examples are supplied, while the fine-tuned parameter persists.

👀 Takeaway: ICL changes the computation for one context; fine-tuning changes the model for future contexts.

### Advanced 4 — Diagnose conflicting demonstrations

**Goal.** Put two similar examples with opposite labels in the prompt, because inconsistent context makes attention-weighted predictions uncertain. We build it in 4 steps.

In [ ]:
x_a4 = np.array([0.9, 1.1, -2.0, 3.0]) # two near-query examples plus distractors.
y_a4 = np.array([1.0, 0.0, 0.0, 1.0]) # nearby examples conflict.
query_a4 = 1.0 # query between conflicting demonstrations.

print("near labels:", y_a4[:2]) # inspect contradiction.

In [ ]:
scores_a4 = rbf_scores(query_a4, x_a4, width=0.4) # sharp attention around the query.
prob_a4, weights_a4 = attention_predict(scores_a4, y_a4) # compute class probability.
uncertainty_a4 = 1 - abs(prob_a4 - 0.5) * 2 # 1 near p=0.5, 0 near p=0 or 1.

print("weights:", np.round(weights_a4, 3), "P(class=1):", round(prob_a4, 3)) # inspect conflict.
print("uncertainty score:", round(uncertainty_a4, 3)) # inspect ambiguity.

In [ ]:
assert 0.35 < prob_a4 < 0.65 # verify conflicting near examples create uncertainty.
major_sources_a4 = np.where(weights_a4 > 0.25)[0] # identify high-weight examples.

print("high-weight conflicting indices:", major_sources_a4) # inspect source of ambiguity.

In [ ]:
plt.figure(figsize=(4.5, 3)); plt.bar(np.arange(len(weights_a4)), weights_a4, color=["green" if v == 1 else "red" for v in y_a4])
plt.title("Advanced 4: conflicting high-weight demos"); plt.xlabel("example"); plt.ylabel("attention weight"); plt.show()

▶ What you'll see: the two conflicting near-query demonstrations both receive high attention, pushing the probability toward 0.5.

👀 Takeaway: ICL can be uncertain or unstable when the prompt contains high-similarity contradictions.

### Advanced 5 — Visualize attention as a kernel smoother

**Goal.** Predict many query points from the same context, because attention regression defines a whole function over inputs. We build it in 4 steps.

In [ ]:
x_ctx_a5 = np.array([-3.0, -1.0, 0.0, 1.0, 3.0]) # fixed prompt examples.
y_ctx_a5 = np.sin(x_ctx_a5) # nonlinear labels.
grid_a5 = np.linspace(-3.5, 3.5, 80) # many query points.

print("context size:", len(x_ctx_a5), "grid size:", len(grid_a5)) # inspect workload.

In [ ]:
preds_a5 = [] # store one ICL prediction per query.
max_weights_a5 = [] # store how concentrated attention is per query.
for q_a5 in grid_a5: # sweep query locations.
    pred_a5, w_a5 = attention_predict(rbf_scores(q_a5, x_ctx_a5, width=0.8), y_ctx_a5) # kernel attention prediction.
    preds_a5.append(pred_a5) # save smooth prediction.
    max_weights_a5.append(float(np.max(w_a5))) # save concentration diagnostic.
preds_a5 = np.array(preds_a5); max_weights_a5 = np.array(max_weights_a5) # convert to arrays.

print("prediction range:", round(float(preds_a5.min()), 3), round(float(preds_a5.max()), 3)) # inspect output scale.

In [ ]:
center_idx_a5 = int(np.argmin(np.abs(grid_a5 - 0.0))) # index near zero query.

print("center prediction:", round(float(preds_a5[center_idx_a5]), 3)) # inspect symmetry around sin(0).

assert abs(float(preds_a5[center_idx_a5])) < 0.1 # verify the smoother is near zero at the center.

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(grid_a5, preds_a5, color="teal", label="ICL smoother")
plt.scatter(x_ctx_a5, y_ctx_a5, color="crimson", zorder=3, label="context examples")
plt.title("Advanced 5: attention regression over many queries"); plt.xlabel("query x"); plt.ylabel("predicted y")
plt.legend(); plt.show()

▶ What you'll see: attention turns a few context examples into a smooth function, with bends near the supplied demonstrations.

👀 Takeaway: across queries, ICL behaves like a prompt-defined function approximator whose shape is controlled by similarity and context examples.